### DistilBERT and CrisisMMD: Calibration and Privacy Budget Experiments

This report evaluates DistilBERT as a model for adaptive differential privacy (DPA-DP) in a simulated disaster scenario, following the earlier calibration work on MobileNetV3.

Without privacy noise, DistilBERT was calibrated on the Emotion dataset (learning rate 0.0001, at least one unfrozen transformer layer, 87-90% accuracy) and then on the text part of CrisisMMD 2.0 (67-69% accuracy, a lower ceiling because the tweets are noisier and the labels more ambiguous).

With DP-SGD enabled, a clipping norm sweep showed that accuracy is unchanged from 0.0001 to 1 and drops only below that, so 0.0001 was used. Four noise-multiplier profiles (one fixed and three DPA-DP) were then compared on a simulated crisis: 180 normal rounds, then 20 rounds with a mix of Emotion and CrisisMMD messages, all with a matched final privacy budget of about epsilon 14.2.

All DPA-DP profiles were slightly worse than the fixed baseline before the crisis but better after it. The profile with the largest phase difference (1.35 normal, 0.65 disaster) performed best, gaining about 2.25 points overall and 5.19 points on disaster messages over the fixed baseline.

Disaster accuracy is dominated by two classes, not_humanitarian and other_relevant_information, which make up 65% of the CrisisMMD train messages (5,166 of 8,000) and account for most of the correct disaster predictions in the confusion matrices averaged over 6 seeds (85% for the fixed profile, 78% for dpa-dp-0.65). Only three of the eight disaster classes are ever predicted correctly; the other five, including missing_or_found_people and vehicle_damage, have no correct predictions. Accuracy therefore mostly reflects the large classes. Macro-F1, which weights all classes equally, is much lower: 20.11 ± 2.13 for the fixed profile and 22.58 ± 0.58 for dpa-dp-0.65 over the disaster labels (95% CI, 6 seeds). The paired difference is +2.47 ± 1.88 points, so the adaptive profile’s advantage persists under macro-F1 but is far smaller than in accuracy. Over all labels the difference (+0.97 ± 1.07) is not significant.

##### page break

In [ ]:
from matplotlib import pyplot
import opacus.accountants
from opacus.accountants import RDPAccountant, PRVAccountant

delta = 1e-5

def epsilon(noise_multiplier_1, noise_multiplier_2, accountant, m=1):
    filename = f"{type(accountant).__name__}_{noise_multiplier_1:.2f}_{noise_multiplier_2:.2f}_{noise_multiplier_switch}_{rounds}_{batch_size}_{n_samples}_{m}.txt"
    try:
        with open(filename) as file:
            return [float(line) for line in file]
    except FileNotFoundError:
        pass
    n_batches = (n_samples * m + batch_size - 1) // batch_size
    sample_rate = m / n_batches
    epsilons = [0]
    for round_idx in range(rounds):
        noise_multiplier = noise_multiplier_1 if round_idx < noise_multiplier_switch else noise_multiplier_2
        for _ in range(n_batches):
            accountant.step(noise_multiplier=noise_multiplier, sample_rate=sample_rate)
        epsilons.append(float(accountant.get_epsilon(delta=delta)))
    # eps, best_alpha = accountant.get_privacy_spent(delta=delta)
    # print(best_alpha)
    with open(filename, "w") as file:
        for value in epsilons:
            file.write(f"{value}\n")
    return epsilons

def plot_epsilon(ax, data, ylabel="epsilon"):
    for name, arr in data.items():
        ax.plot(arr, label=name)
    ax.set_xlabel("round")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend()

In [ ]:
from matplotlib import pyplot
import pandas
def flwr_accuracy_sub(ax, file, name, start=0, key="accuracy", loc=None):
    df = pandas.read_csv(file + ".csv")
    for id_val, g in df.groupby("id", sort=False):
        g = g.sort_values("round").tail(len(g) - start)
        ax.plot(g["round"], g[key], label=str(id_val))

    ax.set_xlabel("round")
    ax.set_ylabel(name)
    ax.grid(True, alpha=0.3)
    if loc is not None:
        ax.legend(loc=loc)

In [ ]:
import ast
import numpy as np
import pandas
from scipy import stats
# FIXME slop

def macrof1():
    df = pandas.read_csv("confusion/confusion.csv")
    df["cm"] = df["confusion"].apply(lambda s: np.array(ast.literal_eval(s), dtype=float))
    
    def cms(profile):
        d = df[df["id"] == profile].sort_values("seed")
        assert len(d) == 6, f"{profile}: expected 6 seeds, got {len(d)}"
        return np.stack(d["cm"].values), d["seed"].values
    
    cms_fixed, seeds_fixed = cms("fixed")
    cms_dpa, seeds_dpa = cms("dpa-dp-0.65")
    assert (seeds_fixed == seeds_dpa).all(), "seed order differs"
    assert cms_fixed.shape[1:] == (14, 14), cms_fixed.shape
    
    def macro_f1(cm, classes):
        tp = np.diag(cm)[classes]
        col = cm[:, classes].sum(0)
        row = cm[classes, :].sum(1)
        prec = np.divide(tp, col, out=np.zeros(len(classes)), where=col > 0)
        rec = np.divide(tp, row, out=np.zeros(len(classes)), where=row > 0)
        s = prec + rec
        f1 = np.divide(2 * prec * rec, s, out=np.zeros(len(classes)), where=s > 0)
        return 100 * f1.mean()  # percentage points
    
    def half_width(x):  # 95% t-interval half-width
        return stats.t.ppf(0.975, len(x) - 1) * np.std(x, ddof=1) / np.sqrt(len(x))
    
    def fmt(x, signed=False):
        m, h = np.mean(x), half_width(x)
        return f"{m:+.2f} ± {h:.2f}" if signed else f"{m:.2f} ± {h:.2f}"
    
    groups = {"All labels": np.arange(14), "Disaster labels": np.arange(6, 14)}
    rows = []
    for name, cls in groups.items():
        f = np.array([macro_f1(c, cls) for c in cms_fixed])
        d = np.array([macro_f1(c, cls) for c in cms_dpa])
        rows.append({"": name, "fixed": fmt(f), "dpa-dp-0.65": fmt(d), "difference": fmt(d - f, signed=True)})
    return pandas.DataFrame(rows).set_index("")

In [ ]:
import pandas
from matplotlib import pyplot
def fixed_diff(data):
    fixed = (data[data["id"] == "fixed"][["seed", "round", "accuracy", "loss"]]
        .rename(columns={"accuracy": "fixed_accuracy", "loss": "fixed_loss",}))
    data_diff = data.merge(fixed, on=["seed", "round"])
    data_diff["accuracy"] = data_diff["accuracy"] - data_diff["fixed_accuracy"]
    data_diff["loss"] = data_diff["loss"] - data_diff["fixed_loss"]
    data_diff = data_diff.drop(columns=["fixed_accuracy", "fixed_loss"])
    return data_diff
def flwr_accuracy_error(ax, file, start=0, key="accuracy"):
    data = pandas.read_csv(file + ".csv")
    data = data[data["round"] >= start]
    data = fixed_diff(data)
    summary = (data.groupby(["id", "round"], sort=False)[key].agg(mean="mean", std="std").reset_index())
    # pyplot.figure(figsize=(11, 6))
    for id_val, group in summary.groupby("id", sort=False):
        group = group.sort_values("round")
        if id_val == "fixed":
            ax.plot(group["round"], group["mean"], label=str(id_val))
        else:
            ax.errorbar(group["round"], group["mean"], yerr=group["std"], linewidth=2, capsize=4, alpha=0.7, label=str(id_val))
    ax.set_xlabel("round")
    ax.set_ylabel(key + " diff")
    ax.grid(True, alpha=0.3)
    ax.legend()
    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin, (ymin + ymax) * 0.7)  # show the lower half
def flwr_accuracy(ax, file, start=0, key="accuracy"):
    data = pandas.read_csv(file + ".csv")
    data = data[data["round"] >= start]
    summary = (data.groupby(["id", "round"], sort=False)[key].agg(mean="mean", std="std").reset_index())
    for id_val, group in summary.groupby("id", sort=False):
        group = group.sort_values("round")
        line, = ax.plot(group["round"], group["mean"], label=str(id_val))
        ax.fill_between(
        group["round"],
        group["mean"] - group["std"],
        group["mean"] + group["std"],
        color=line.get_color(),
        alpha=0.2
    )

    ax.set_xlabel("round")
    ax.set_ylabel(key)
    ax.grid(True, alpha=0.3)
    ax.legend()

In [ ]:
def accuracy_summary(file):
    data = pandas.read_csv(file + ".csv")
    data = data[data["round"] > 190]
    fixed = (
        data[data["id"] == "fixed"]
        [["seed", "round", "accuracy"]]
        .rename(columns={"accuracy": "fixed"})
    )
    data = data.merge(fixed, on=["seed", "round"], how="left")
    data["fixed"] = data["accuracy"] - data["fixed"]
    summary = (data.groupby("id", sort=False)["accuracy"].agg(mean="mean", deviation="std").reset_index())
    summary_diff = (data.groupby("id", sort=False)["fixed"].agg(diff="mean", dev_diff="std").reset_index())
    summary = summary.merge(summary_diff, on=["id"], how="left")
    summary = summary.set_index("id")
    summary.index.name = None
    summary = summary.style.format("{:.4f}")
    return summary

In [ ]:
# accuracy_summary("profiles/flwr6_all")

In [ ]:
# accuracy_summary("profiles/flwr6_sub")

In [ ]:
def flwr_accuracy_fill(ax, file, name, start=0, key="accuracy", loc=None):
    data = pandas.read_csv(file + ".csv")
    data = data[data["round"] >= start]
    summary = (data.groupby(["id", "round"], sort=False)[key].agg(mean="mean", std="std").reset_index())
    for id_val, group in summary.groupby("id", sort=False):
        group = group.sort_values("round")
        line, = ax.plot(group["round"], group["mean"], label=str(id_val))
        ax.fill_between(
        group["round"],
        group["mean"] - group["std"],
        group["mean"] + group["std"],
        color=line.get_color(),
        alpha=0.2
    )

    ax.set_xlabel("round")
    ax.set_ylabel(name)
    ax.grid(True, alpha=0.3)
    if loc is not None:
        ax.legend(loc=loc)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
# FIXME slop

def advantage_table(metric="accuracy", last=10, folder=".", decimals=2, scale=100):
    """
    Noise-switch gain (profile - fixed), per scenario, plus
    interaction = phase gain - no-phase gain.
    Per-seed value = mean over the last `last` rounds; everything is paired by seed.
    Cell = "mean ± half-width" of the 95% t-interval over seeds.
    Accuracy is shown in percentage points; loss is left in raw units.
    """

    def load(prefix, suffix):
        df = pd.read_csv(f"{prefix}/flwr6_{suffix}.csv")
        df = df[df["round"] > df["round"].max() - last]
        return df.groupby(["seed", "id"])[metric].mean().unstack("id")

    def cell(x):
        n = len(x)
        m = x.mean()
        h = stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n)
        return f"{m:+.{decimals}f} ± {h:.{decimals}f}"

    groups = {"All labels": "all", "Disaster labels": "sub"}
    quantities = ["Event gain", "No-event", "Advantage"]
    data = {}

    for glabel, suffix in groups.items():
        ph = load("profiles", suffix).dropna()
        npz = load("nophase", suffix).dropna()
        seeds = ph.index.intersection(npz.index)
        ph, npz = ph.loc[seeds], npz.loc[seeds]
        # print(f"{glabel}: {len(seeds)} matched seeds -> {list(seeds)}")

        profiles = sorted((c for c in ph.columns if c != "fixed"), reverse=True)
        for prof in profiles:
            g_ph = scale * (ph[prof] - ph["fixed"]).values
            g_np = scale * (npz[prof] - npz["fixed"]).values
            for q, x in zip(quantities, [g_ph, g_np, g_ph - g_np]):
                data.setdefault((glabel, q), {})[prof] = cell(x)

    summary = pd.DataFrame(data)
    summary.columns = pd.MultiIndex.from_tuples(summary.columns)
    summary.index.name = None
    return summary

print(advantage_table().to_latex().replace("±", r"$\pm$"))

In [ ]:
import pandas
from matplotlib import pyplot
import numpy
def confusion_matrix(profile, ax0=None):
    data = pandas.read_csv("confusion/confusion.csv")
    data["confusion"] = data["confusion"].apply(eval)
    cm0 = data[(data["id"] == profile) & (data["seed"] == 0)].iloc[0]["confusion"]
    cm1 = data[(data["id"] == profile) & (data["seed"] == 1)].iloc[0]["confusion"]
    cm2 = data[(data["id"] == profile) & (data["seed"] == 2)].iloc[0]["confusion"]
    cm3 = data[(data["id"] == profile) & (data["seed"] == 3)].iloc[0]["confusion"]
    cm4 = data[(data["id"] == profile) & (data["seed"] == 4)].iloc[0]["confusion"]
    cm5 = data[(data["id"] == profile) & (data["seed"] == 5)].iloc[0]["confusion"]
    cm = numpy.rint(numpy.mean([cm0, cm1, cm2, cm3, cm4, cm5], axis=0)).astype(int)
    ax = pyplot.subplots(figsize=(5, 4))[1] if ax0 is None else ax0
    image = ax.imshow(cm, cmap="Blues")
    for row in range(len(cm)):
        for col in range(len(cm[row])):
            ax.text(col, row, cm[row][col], ha="center", va="center", # if cm[row][col] > 0 or row == col else " "
                color="white" if cm[row][col] > max(max(row) for row in cm) / 2 else "black")
    ax.set_title("confusion matrix for " + profile + " profile")
    ax.set_xticks([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13])
    ax.set_yticks([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13])
    ax.set_xticklabels(["s", "j", "l", "a", "f", "s", "a", "i", "i", "m", "n", "o", "r", "v"])
    ax.set_yticklabels(["sadness", "joy", "love", "anger", "fear", "surprise",
        "affected", "infrastr", "injured","missing",
        "not", "other", "rescue", "vehicle"])
    if ax0 is None:
        pyplot.tight_layout()
        pyplot.show()

##### page break

### DistilBERT performance by learning rate (August)
noise_multiplier: 0, tune_layers: 1, batch_size: 32

DistilBERT model on the Emotion dataset (16,000 training / 2,000 test texts) with privacy noise disabled. Accuracy and loss both improved steadily as the learning rate increased from 1e-7 to 0.001, but at 0.01 and above the model stopped learning, producing effectively random output. Learning rates of 0.001 and 0.0001 were selected for the following experiments as they give good results.

In [ ]:
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(13, 4))
flwr_accuracy_sub(ax1, "../05_CrisisMMD/bert_clear_learning", "accuracy")
flwr_accuracy_sub(ax2, "../05_CrisisMMD/bert_clear_learning", "loss", key="loss", loc="upper right")
# fig.savefig("text_lr_sweep.pdf", bbox_inches="tight")

##### page break

### DistilBERT performance by frozen layers (August)
noise_multiplier: 0, learning_rate: 0.0001, batch_size: 32

With layers 2, 3 and above unfrozen, the model converges to 88-90% accuracy on the Emotion dataset. However, a fully frozen backbone with only the classifier head trainable, was unable to provide result better than 60%, indicating that at least one unfrozen backbone layer is necessary for DistilBERT performance.

In [ ]:
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(13, 4))
flwr_accuracy_sub(ax1, "../05_CrisisMMD/bert_clear_layers", "accuracy")
flwr_accuracy_sub(ax2, "../05_CrisisMMD/bert_clear_layers", "loss", key="loss", loc="upper right")
# fig.savefig("text_layers_emotion.pdf", bbox_inches="tight")

##### page break

### DistilBERT performance on CrisisMMD (August)
noise_multiplier: 0, learning_rate: 0.0001, batch_size: 32

In this test the Emotion dataset was replaced with the text modality of CrisisMMD 2.0, a state of the art dataset of social media messages and images from various disaster types. It defines 8 labels and is pre-split by its authors into 13,608 training and 2,237 test texts.

Configurations with layer 1 and above unfrozen converge to 67-69% accuracy. Models with more unfrozen layers show a higher rise in loss value during later training rounds.

In [ ]:
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(13, 4))
flwr_accuracy_sub(ax1, "../05_CrisisMMD/crisismmd_layers", "accuracy")
flwr_accuracy_sub(ax2, "../05_CrisisMMD/crisismmd_layers", "loss", key="loss", loc="upper right")
# fig.savefig("text_layers_crisismmd.pdf", bbox_inches="tight")

##### page break

### DistilBERT DPA-DP privacy budget allocation
Epsilon budget consumption over 200 rounds (180 normal + 20 simulated disaster phase), for four noise-multiplier profiles tuned to reach a matched final budget of PRV epsilon around 14.2.
- fixed - uniform noise multiplier 1.0 across both phases
- dpa-dp-0.85 - low profile: 1.03 normal, 0.85 disaster
- dpa-dp-0.75 - medium profile: 1.08 normal, 0.75 disaster
- dpa-dp-0.65 - high profile: 1.35 normal, 0.65 disaster, the largest phase difference

In [ ]:
rounds=200
noise_multiplier_switch = 180
batch_size = 32
n_samples = 1600 # emotion 16000 training set / 10 clients

prv = {}
prv["fixed"] = epsilon(1.00, 1.00, PRVAccountant())
prv["dpa-dp-0.85"] = epsilon(1.03, 0.85, PRVAccountant())
prv["dpa-dp-0.75"] = epsilon(1.08, 0.75, PRVAccountant())
prv["dpa-dp-0.65"] = epsilon(1.35, 0.65, PRVAccountant())
print({k: a[-1] for k, a in prv.items()})
rdp = {}
rdp["fixed"] = epsilon(1.00, 1.00, RDPAccountant())
rdp["dpa-dp-0.85"] = epsilon(1.03, 0.85, RDPAccountant())
rdp["dpa-dp-0.75"] = epsilon(1.08, 0.75, RDPAccountant())
rdp["dpa-dp-0.65"] = epsilon(1.35, 0.65, RDPAccountant())
print({k: a[-1] for k, a in rdp.items()})
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(13, 4))
plot_epsilon(ax1, prv, "PRV epsilon")
plot_epsilon(ax2, rdp, "RDP epsilon")
# fig.savefig("text_epsilon.pdf", bbox_inches="tight")

##### page break

### Text dataset for crisis simulation
Similarly to the MobileNetV3 experiment, we created a dataset containing normal messages for the normal stage (16,000 texts from the Emotion dataset). After round 180 (crisis onset), these were replaced with a mix of normal and disaster messages (8,000 Emotion + 8,000 CrisisMMD v2.0). The detailed labels and text counts are given in the table below. Accuracy was evaluated on the merged test splits of both datasets.

In [ ]:
df = pandas.read_csv("dataset_size.csv")
display(df)

##### page break

### DistilBERT accuracy by clipping norm
tune_layers: 2, learning_rate: 0.001, batch_size: 32, noise_multiplier: 1

optimizer: AdamW (Ilya Loshchilov and Frank Hutter. Decoupled Weight Decay Regularization. arXiv preprint arXiv:1711.05101, 2017)

We ran 15 rounds on the combined dataset (8,000 Emotion + 8,000 CrisisMMD) with the fixed profile (sigma = 1) and different max_grad_norm values (the DP-SGD clipping norm). Accuracy decreases when the value falls below 0.0001, but it remains unchanged across a wide range from 0.0001 to 1. The flat range is consistent with the behavior of Adam-type optimizers under clipping: when all per-sample gradients are clipped, the signal and the noise both scale with C, and the normalised update does not change.

In [ ]:
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(13, 4))
flwr_accuracy_sub(ax1, "norm/norm10_all", "overall accuracy", 5)
flwr_accuracy_sub(ax2, "norm/norm10_sub", "disaster accuracy", 5, loc="upper left")
# fig.savefig("text_clip_sweep.pdf", bbox_inches="tight")

##### page break

### Accuracy between privacy profiles
tune_layers: 2, learning_rate: 0.001, batch_size: 32, max_grad_norm: 0.0001

Two unfrozen layers plus the classifier were chosen as a middle ground between reasonable accuracy and a reduced number of parameters exposed to the privacy noise. During normal rounds, all DPA-DP profiles show slightly worse but comparable accuracy to the fixed baseline. After the simulated disaster onset at round 180, all profiles show better accuracy than the fixed baseline, with the high profile performing best.

In [ ]:
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(13, 4))
flwr_accuracy_fill(ax1, "profiles/flwr6_all", "overall accuracy", 160)
flwr_accuracy_error(ax2, "profiles/flwr6_all", 160)
# fig.savefig("text_profiles_accuracy.pdf", bbox_inches="tight")

In [ ]:
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(13, 4))
flwr_accuracy_fill(ax1, "profiles/flwr6_sub", "disaster accuracy", 160)
flwr_accuracy_fill(ax2, "profiles/flwr6_all", "overall loss", 160, key="loss", loc="upper right")
# fig.savefig("text_profiles_disaster_loss.pdf", bbox_inches="tight")

##### page break

In [ ]:
fig, (ax1, ax2) = pyplot.subplots(1, 2, figsize=(11, 5))
confusion_matrix("fixed", ax1)
confusion_matrix("dpa-dp-0.65", ax2)
# fig.savefig("text_confusion_full.pdf", bbox_inches="tight")

Confusion matrix (6 seeds) for the fixed sigma=1.00 profile and the high 0.65 profile.

In [ ]:
macrof1()

Macro-F1 in percentage points at round 200. “All labels” averages the per-class F1 over all 14 classes, “Disaster labels” over the 8 CrisisMMD classes. ± is the half-width of the 95% t-interval over 6 matched seeds; the difference is dpa-dp-0.65 minus fixed, computed per seed.

##### page break

### No-event control

Each profile was run in two scenarios with the same noise schedule (switch at round 180). In the event scenario, the model is trained on Emotion texts for 180 rounds and then on the combined Emotion and CrisisMMD data for 20 rounds. In the no-event control, it is trained on the combined data from round 1. For each seed, the accuracy difference to the fixed profile (σ = 1) is averaged over rounds 191-200. Event gain is this difference in the event scenario, no-event is the same difference in the control, and advantage is event gain minus no-event.

In [ ]:
advantage_table()

Accuracy difference to the fixed profile, in percentage points, averaged over rounds 191-200. ± is the half-width of the 95% t-interval over 6 matched seeds.